# AI-F-001 · 모델 및 API 통합 검증

이 노트북은 **앞서 진행한 F-001 검증(규칙 기반 → 직무군 ML → 작업 특성 ML/DL → FastAPI → pytest)**을 한곳에서 재현합니다.

- 실험용 키워드 후보 라벨은 정답 데이터가 아닙니다.
- 모델 점수는 실제 업무 위험도의 확률이 아닙니다.
- `artifacts/`의 모델은 로컬에 있어야 하며, 저장소에 포함되지 않을 수 있습니다.
- 실행 결과는 저장하지 않았습니다. 각 셀을 직접 실행하세요.

## 1. 프로젝트 루트 탐색 및 실행 환경

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / 'src' / 'features' / 'f001_job_task').is_dir()), None)
if root is None:
    raise RuntimeError('ai/ 프로젝트 디렉터리에서 Jupyter를 실행하세요.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
feature = root / 'src/features/f001_job_task'
artifacts = feature / 'artifacts'
print('AI 루트:', root)
print('기능 폴더:', feature)

## 2. 모델·데이터·평가 결과 파일 확인

In [ ]:
paths = {
    '직무군 ML': artifacts / 'models/f001_best_weak.joblib',
    '직무군 Weak': artifacts / 'models/job_classifier_weak.joblib',
    '작업 특성 ML': artifacts / 'models/task_multilabel_weak.joblib',
    '작업 특성 DL': artifacts / 'comparison/task_cnn_best.pt',
    '모델 비교': artifacts / 'comparison/metrics.csv',
    '예측 비교': artifacts / 'comparison/predictions.csv',
}
report = pd.DataFrame([{'대상': k, '경로': str(v.relative_to(feature)), '존재': v.is_file()} for k,v in paths.items()])
display(report)
if not all(report.loc[:3, '존재']):
    print('경고: 필수 모델이 없습니다. 먼저 로컬 artifacts 경로를 확인하세요.')

## 3. 규칙 기반 베이스라인 테스트 (모델 파일 불필요)

In [ ]:
from src.features.f001_job_task.model import JobInput, analyze_job, verify_evidence
job = JobInput(jobId=1, jobTitle='환경미화', description='야외 공원 청소 및 도보 작업', workHours=8)
result = analyze_job(job)
assert result['reviewRequired'] is True
assert verify_evidence(job.description, result['evidence'])
display(pd.DataFrame(result['evidence']))
print('규칙 기반 태그:', result['taskTags'])

## 4. 직무군 분류 + 작업 특성 ML/DL 통합 예측

In [ ]:
from src.features.f001_job_task.inference import predict_f001
from pprint import pprint

samples = ['야간 배송기사 모집', '아파트 경비원 모집', '요양보호사 채용']
records = []
for title in samples:
    for kind in ('ml', 'dl'):
        try:
            result = predict_f001(title, task_model=kind)
            assert result['feature_id'] == 'AI-F-001'
            assert result['task_analysis']['selected_model'] == kind
            assert isinstance(result['task_analysis']['task_characteristics'], list)
            records.append({'title': title, 'model': kind, 'job': result['job_classification']['category'], 'task_count': len(result['task_analysis']['task_characteristics']), 'status':'PASS'})
        except (FileNotFoundError, RuntimeError, ValueError) as exc:
            records.append({'title': title, 'model': kind, 'job': None, 'task_count': None, 'status':f'ERROR: {exc}'})
display(pd.DataFrame(records))

## 5. 모델 비교 CSV 읽기 및 시각화 (파일이 있을 때만)

In [ ]:
metrics_path = artifacts / 'comparison/metrics.csv'
if metrics_path.is_file():
    metrics = pd.read_csv(metrics_path)
    display(metrics.head(20))
    cols = metrics.select_dtypes(include='number').columns
    if len(cols):
        metrics[list(cols)].plot(kind='bar', figsize=(11,4), title='AI-F-001 모델 평가 지표 (저장된 실험 결과)')
else:
    print('metrics.csv 없음: 모델 비교 시각화 생략')

## 6. FastAPI 통합 테스트 (ML/DL)

In [ ]:
from fastapi.testclient import TestClient
from src.api.main import app

client = TestClient(app)
for kind in ('ml', 'dl'):
    response = client.post('/api/v1/jobs/analyze-integrated', json={'title':'야간 배송기사 모집','task_model':kind})
    print(kind, 'HTTP', response.status_code)
    if response.status_code == 200:
        payload = response.json()
        assert payload['feature_id'] == 'AI-F-001'
        assert payload['task_analysis']['selected_model'] == kind
    else:
        print(str(response.text)[:700])

## 7. 기존 pytest 실행 (모든 기능별 테스트)

테스트는 `features/f001_job_task/tests/`에 **별도 파일로 계속 보존**합니다. 여기서는 결과를 재현만 합니다.

> 원격 GitHub 브랜치에 테스트 디렉터리가 없을 수 있습니다. 로컬에 있는 27개 테스트를 실행하세요.

In [ ]:
import subprocess
import sys
suite = feature / 'tests'
if not suite.is_dir():
    print('로컬에 F-001 tests/ 폴더가 없습니다. tests 파일을 기능 폴더로 복원한 후 다시 실행하세요.')
else:
    proc = subprocess.run([sys.executable, '-m', 'pytest', str(suite), '-q'], cwd=root, text=True, capture_output=True)
    print(proc.stdout[-6000:])
    if proc.stderr:
        print(proc.stderr[-1500:])
    print('pytest exit code:', proc.returncode)
    assert proc.returncode == 0, 'F-001 테스트 실패: 출력 확인'

## 8. 평가 해석

- **pytest 통과**: 변경 후 기능 회귀 여부 확인
- **ML/DL 응답 확인**: FastAPI 요청 → 직무군/작업특성 예측 흐름 확인
- **비교 CSV**: 학습 당시 실험 기록을 읽음. 정답 데이터 기반 평가라고 단정할 수 없음
- **배포 이전**: 모델 아티팩트 경로·버전 관리, 독립 정답 검수 데이터 검증 필요